# Exercício 2 — Item B: Pipeline Integrado Sequencial

**Aluno:** Thiago Costa  
**Avaliação:** DR2 — AT  

Este notebook integra o pipeline completo dos trabalhos práticos anteriores em um fluxo sequencial único:
1. Captura de frame e correção de distorção (`undistort` — Exercício 1)
2. Segmentação de ROI por cor HSV (TP1)
3. Extração de descritores ORB na ROI (TP2)
4. Detecção clássica HOG+SVM e Haar Cascade (TP3)
5. Classificação da ROI via OpenCV DNN (Item A)

O tempo de processamento de cada etapa é cronometrado e exibido no frame final.

In [ ]:
import time
import cv2
import numpy as np
from pathlib import Path
from dnn_utils import (
    read_pipeline_frame, DEFAULT_K, DEFAULT_DIST,
    segment_red_hsv, orb_features, hog_or_haar_detector,
    predict_opencv, topk, draw_top3, load_labels, OUT, ensure_dirs
)

ensure_dirs()
tempos = {}

In [ ]:
# ETAPA 1 — Captura e Correção de Distorção (undistort)
frame, origem = read_pipeline_frame()
t0 = time.perf_counter()
frame_corrigido = cv2.undistort(frame, DEFAULT_K, DEFAULT_DIST)
tempos["undistort"] = (time.perf_counter() - t0) * 1000
print(f"undistort: {tempos['undistort']:.2f} ms")

In [ ]:
# ETAPA 2 — Segmentação de ROI por cor HSV
t0 = time.perf_counter()
mask, roi_hsv = segment_red_hsv(frame_corrigido)
tempos["HSV"] = (time.perf_counter() - t0) * 1000

vis = frame_corrigido.copy()
if roi_hsv:
    x, y, w, h = roi_hsv
    roi_img = frame_corrigido[y:y+h, x:x+w]
    cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 255, 255), 3)
    cv2.putText(vis, "ROI HSV", (x, y-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 120, 120), 2)
else:
    roi_img = frame_corrigido
    x, y, w, h = 0, 0, frame_corrigido.shape[1], frame_corrigido.shape[0]
print(f"HSV: {tempos['HSV']:.2f} ms")

In [ ]:
# ETAPA 3 — Extração de Features ORB
t0 = time.perf_counter()
kp, des = orb_features(roi_img)
tempos["ORB"] = (time.perf_counter() - t0) * 1000
for p in kp[:60]:
    px, py = int(p.pt[0] + x), int(p.pt[1] + y)
    cv2.circle(vis, (px, py), 3, (0, 255, 0), -1)
print(f"ORB: {tempos['ORB']:.2f} ms")

In [ ]:
# ETAPA 4 — Detecção Clássica HOG+SVM e Haar Cascade
t0 = time.perf_counter()
boxes = hog_or_haar_detector(frame_corrigido)
tempos["HOG_Haar"] = (time.perf_counter() - t0) * 1000
for nome, bx, by, bw, bh in boxes:
    cv2.rectangle(vis, (bx, by), (bx+bw, by+bh), (255, 0, 0), 2)
    cv2.putText(vis, nome, (bx, by-6), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 0, 0), 2)
print(f"HOG/Haar: {tempos['HOG_Haar']:.2f} ms")

In [ ]:
# ETAPA 5 — Classificação da ROI com OpenCV DNN
t0 = time.perf_counter()
prob = predict_opencv(roi_img)
tempos["DNN"] = (time.perf_counter() - t0) * 1000
top3 = topk(prob, 3)
labels = load_labels()
vis = draw_top3(vis, top3, labels)
print(f"DNN: {tempos['DNN']:.2f} ms")

In [ ]:
# ETAPA 6 — Visualização Final com Anotação de Tempos
y0 = vis.shape[0] - 140
cv2.rectangle(vis, (10, y0), (380, vis.shape[0] - 10), (255, 255, 255), -1)
cv2.rectangle(vis, (10, y0), (380, vis.shape[0] - 10), (0, 0, 0), 1)
for i, (nome, ms) in enumerate(tempos.items()):
    cv2.putText(vis, f"{nome}: {ms:.1f} ms", (22, y0 + 24 + i*20), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (0, 0, 0), 2)
total_ms = sum(tempos.values())
cv2.putText(vis, f"Total: {total_ms:.1f} ms", (22, y0 + 24 + len(tempos)*20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 180), 2)
saida = OUT / "item_b_pipeline_integrado.jpg"
cv2.imwrite(str(saida), vis)
print(f"Total: {total_ms:.2f} ms. Frame salvo em: {saida}")